# 01 - 数据分析

本笔记本用于探索和理解金融时序数据的特性，为后续的特征工程和模型训练做准备。

## 1. 导入必要的库

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import DataLoader
from src.utils.visualization import plot_time_series, plot_ohlc

%matplotlib inline
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

## 2. 加载数据

In [ ]:
# 初始化数据加载器
loader = DataLoader(time_column='timestamp')

# 加载样例数据
df = loader.load('../data/sample/sample_data.csv')

print(f"数据形状: {df.shape}")
print(f"\n数据列: {df.columns.tolist()}")
print(f"\n数据类型:\n{df.dtypes}")
print(f"\n前5行:\n{df.head()}")

## 3. 数据概览

In [ ]:
# 基本统计信息
print("基本统计信息:")
df.describe()

In [ ]:
# 检查缺失值
print("缺失值统计:")
print(df.isnull().sum())

In [ ]:
# 水印标签分布
if 'watermark_label' in df.columns:
    print("\n水印标签分布:")
    print(df['watermark_label'].value_counts())
    print(f"\n水印比例: {df['watermark_label'].mean():.2%}")

## 4. 可视化分析

In [ ]:
# 价格走势
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# OHLC价格
axes[0, 0].plot(df.index, df['close'], label='Close', linewidth=2)
axes[0, 0].fill_between(df.index, df['low'], df['high'], alpha=0.3, label='Range')
axes[0, 0].set_title('Price Movement')
axes[0, 0].set_xlabel('Time')
axes[0, 0].set_ylabel('Price')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 成交量
axes[0, 1].bar(df.index, df['volume'], alpha=0.7, color='orange')
axes[0, 1].set_title('Volume')
axes[0, 1].set_xlabel('Time')
axes[0, 1].set_ylabel('Volume')
axes[0, 1].grid(True, alpha=0.3)

# 收益率分布
returns = df['close'].pct_change().dropna()
axes[1, 0].hist(returns, bins=30, alpha=0.7, edgecolor='black')
axes[1, 0].set_title('Returns Distribution')
axes[1, 0].set_xlabel('Return')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].axvline(returns.mean(), color='r', linestyle='--', label=f'Mean: {returns.mean():.4f}')
axes[1, 0].legend()

# 水印标记
if 'watermark_label' in df.columns:
    axes[1, 1].plot(df.index, df['close'], label='Close', alpha=0.7)
    watermark_points = df[df['watermark_label'] == 1]
    axes[1, 1].scatter(watermark_points.index, watermark_points['close'], 
                      color='red', label='Watermark', s=50, zorder=5)
    axes[1, 1].set_title('Watermark Locations')
    axes[1, 1].set_xlabel('Time')
    axes[1, 1].set_ylabel('Price')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. 时序特性分析

In [ ]:
# 自相关分析
from pandas.plotting import autocorrelation_plot

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 价格自相关
autocorrelation_plot(df['close'], ax=axes[0])
axes[0].set_title('Price Autocorrelation')

# 收益率自相关
autocorrelation_plot(returns, ax=axes[1])
axes[1].set_title('Returns Autocorrelation')

plt.tight_layout()
plt.show()

In [ ]:
# 滚动统计
window = 5
rolling_mean = df['close'].rolling(window=window).mean()
rolling_std = df['close'].rolling(window=window).std()

plt.figure(figsize=(12, 6))
plt.plot(df.index, df['close'], label='Close', alpha=0.7)
plt.plot(df.index, rolling_mean, label=f'Rolling Mean ({window})', linewidth=2)
plt.fill_between(df.index, 
                 rolling_mean - 2*rolling_std, 
                 rolling_mean + 2*rolling_std, 
                 alpha=0.2, label='±2 Std')
plt.title('Rolling Statistics')
plt.xlabel('Time')
plt.ylabel('Price')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 6. 数据质量检查

In [ ]:
# 异常值检测
from scipy import stats

z_scores = np.abs(stats.zscore(df['close']))
outliers = df[z_scores > 3]

print(f"检测到 {len(outliers)} 个异常值 (|z-score| > 3)")

if len(outliers) > 0:
    print("\n异常值:")
    print(outliers)

In [ ]:
# 数据完整性检查
time_diff = df.index.to_series().diff().dropna()
print(f"时间间隔统计:")
print(time_diff.describe())

# 检查是否有缺失的时间点
expected_freq = time_diff.mode()[0]
expected_range = pd.date_range(start=df.index[0], end=df.index[-1], freq=expected_freq)
missing_times = expected_range.difference(df.index)

print(f"\n期望频率: {expected_freq}")
print(f"缺失时间点数量: {len(missing_times)}")

if len(missing_times) > 0:
    print(f"\n前10个缺失时间点:")
    print(missing_times[:10])

## 7. 总结

通过上述分析，我们可以得出以下结论：

1. **数据概况**: 了解数据的基本统计特性
2. **数据质量**: 检查缺失值、异常值等
3. **时序特性**: 分析自相关性、趋势等
4. **水印分布**: 了解水印标签的分布情况

这些分析结果将指导我们进行特征工程和模型选择。